In [2]:
import json

In [3]:
def reduce_results():
    
    with open("results.json") as f:
        results = json.load(f)
    
    def get_cruk_tcga(t, h):
        res = results[t][h]
        terms = [r.get('CRUK', []) + r.get('TCGA', []) for r in res['results']]
        extra = []
        for term in terms:
            for t in term:
                if t not in extra:
                    extra.append(t)
        return extra
        
    reduced = {}
    topographies = list(results.keys())
    histologies = list(results[topographies[0]].keys())
    for t in topographies:
        reduced[t] = {}
        for h in histologies:
            extra = get_cruk_tcga(t,h)
            if extra:
                reduced[t][h] = extra

    with open("extra_terms.json", "w") as f:
        json.dump(reduced, f, indent=3)

#reduce_results()

def get_dummy(i):
    new_dummies_path = "../frontend/CRUK_datahub_landing_page/src/utils/new_dummies"
    with open(f"{new_dummies_path}/dataset_{str(i).zfill(2)}.json") as f:
        dummy = json.load(f)
    return dummy

with open("extra_terms.json") as f:
    extra_terms = json.load(f)

def get_extra_terms(dummy, extra_terms):
    filters = dummy["datasetFilters"]
    tops = [f["label"] for f in filters if f["id"][:5] == "0_0_0"]
    hist = [f["label"] for f in filters if f["id"][:5] == "0_0_1"]
    
    extra = []
    ids = set()
    for t in tops:
        for h in hist:
            e = extra_terms[t][h]
            for term in e:
                if term["id"] not in ids:
                    extra.append(term)
                    ids |= {term["id"]}
    return extra

def get_child_terms(extra, dummy, ids):
    
    try:
        if dummy["coverage"]["typicalAgeRangeMax"] < 19 or "0_2_3_0_0" in ids: # we have a young person or child
            
            extra.append({'id': '0_0_2_23',
                          'label': "Children's cancers",
                          'category': 'crukTerms',
                          'primaryGroup': 'cancer-type',
                          'description': ''})
            
            if "0_0_2_1" in ids: # Acute lymphoblastic leukaemia 
                extra.append({'id': '0_0_2_2',
                              'label': 'Acute lymphoblastic leukaemia (ALL) in children',
                              'category': 'crukTerms',
                              'primaryGroup': 'cancer-type',
                              'description': ''})
            if "0_0_2_12" in ids: # Brain tumours
                extra.append({'id': '0_0_2_13',
                              'label': 'Brain tumours in children',
                              'category': 'crukTerms',
                              'primaryGroup': 'cancer-type',
                              'description': ''})
            if "0_0_2_70" in ids: # Non-Hodgkin lymphoma
                extra.append({'id': '0_0_2_71',
                              'label': 'Non-Hodgkin lymphoma in children',
                              'category': 'crukTerms',
                              'primaryGroup': 'cancer-type',
                              'description': ''})
            return extra
    except:
        return extra
    

def get_male_specific(extra, dummy, ids):

    try:
        keywords = dummy["summary"]["keywords"]
    except:
        return extra
        
    lower_keywords = [i.lower() for i in keywords]
    if "0_0_2_12" not in ids:
        for phrase in lower_keywords:
            for m,f in [("male","female"), ("men","women"), ("man","woman"), ("boy", "girl")]:
                if m in phrase and f not in phrase:
                    extra.append({'id': '0_0_2_13',
                                          'label': "Men's cancer",
                                          'category': 'crukTerms',
                                          'primaryGroup': 'cancer-type',
                                          'description': ''})  
                    if "0_0_2_14" in ids:
                        extra.append({'id': '0_0_2_15',
                                          'label': "Breast cancer in men",
                                          'category': 'crukTerms',
                                          'primaryGroup': 'cancer-type',
                                          'description': ''})
                return extra

def get_extra(dummy, extra_terms):
    tcga_cruk = get_extra_terms(dummy, extra_terms)
    ids = set([e["id"] for e in tcga_cruk])
    plus_children = get_child_terms(tcga_cruk, dummy, ids)
    plus_men = get_male_specific(plus_children, dummy, ids)
    return plus_men


with open("male_child_breast_cancer.json") as f:
    dummy = json.load(f)

dummy

{'coverage': {'typicalAgeRangeMax': 15, 'typicalAgeRangeMin': 3},
 'summary': {'keywords': ['men', 'Males', 'boys'],
  'title': 'male child dummy breast cancer',
  'populationSize': 0,
  'doiName': None,
  'datasetAliases': None},
 'datasetFilters': [{'id': '0_0_0_9_1',
   'label': 'C50.1 Central portion of breast',
   'category': 'C50 Breast',
   'primaryGroup': 'cancer-type',
   'description': ''},
  {'id': '0_0_1_9',
   'label': '850-854 Ductal and lobular neoplasms',
   'category': 'icdOHistology',
   'primaryGroup': 'cancer-type',
   'description': ''},
  {'id': '0_2_3_0_0',
   'label': 'Child and Young Person',
   'category': 'Background',
   'primaryGroup': 'data-type',
   'description': ''}],
 'icons': ['Patient Study', 'Background Information'],
 'modified': '2026-05-19T14:33:44.244Z'}

In [104]:
extra = get_extra(dummy, extra_terms)

KeyError: '850-854 Ductal and lobular neoplasms'

In [6]:
with open("label_key_dict.json") as f:
    lkd = json.load(f)

In [11]:
lkd_hist = [i for i in lkd if lkd[i][:5] == "0_0_1"][1:]

In [25]:
lkd_top = [i for i in lkd if lkd[i][:5] == "0_0_0"][1:]

In [12]:
lkd_hist[:3]

['800 Neoplasms, NOS',
 '8000/0 Neoplasm, benign',
 '8000/1 Neoplasm, uncertain whether benign or malignant']

In [23]:
topographies = list(extra_terms.keys())

In [24]:
histologies = list(extra_terms[topographies[0]].keys())

In [20]:
with open("missing_histologies.json", "w") as f:
    json.dump([i for i in lkd_hist if i not in histologies], f, indent=3)

In [27]:
with open("missing_topographies.json", "w") as f:
    json.dump([i for i in lkd_top if i not in topographies], f, indent=3)